# 01 · Catalog tracks and every credited artist
The catalog is the base discovery collection. Keep valid Spotify track IDs even when audio fields are missing; represent invalid audio values explicitly. Parse artist lists safely and retain all credits, including collaborations.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/data_pipeline.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Image
import pandas as pd
from scripts.data_pipeline import OUT, REPORTS


In [2]:
from scripts.data_pipeline import catalog
quality = catalog()

{
  "source_tracks": 586672,
  "clean_tracks": 586672,
  "artist_metadata_rows": 1162095,
  "track_artist_edges": 757170,
  "tracks_with_multiple_artists": 106916,
  "invalid_audio_values_preserved_as_missing": {
    "danceability": 0,
    "energy": 0,
    "loudness": 0,
    "speechiness": 0,
    "acousticness": 0,
    "instrumentalness": 0,
    "liveness": 0,
    "valence": 0,
    "tempo": 328,
    "duration_ms": 0,
    "key": 0,
    "mode": 0
  },
  "country_column_in_source": false
}


In [3]:
tracks = pd.read_parquet(OUT / "catalog_tracks.parquet")
display(tracks[["id", "name", "artists", "id_artists"]].head(10))
display(pd.read_parquet(OUT / "catalog_track_artists.parquet").head(10))

,id,name,artists,id_artists
0,35iwgR4jXetI318WEWsa1Q,Carve,[Uli],[45tIt06XoI0Iio4LBEVpls]
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,[Fernando Pessoa],[14jtPCOoNZwquk5wd9DxrY]
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,[Ignacio Corsini],[5LiOoJbxVSAMkBS2fUm3X2]
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,[Ignacio Corsini],[5LiOoJbxVSAMkBS2fUm3X2]
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,[Dick Haymes],[3BiJGZsyX9sJchTqcSA7Su]
5,0BRXJHRNGQ3W4v9frnSfhu,Ave Maria,[Dick Haymes],[3BiJGZsyX9sJchTqcSA7Su]
6,0Dd9ImXtAtGwsmsAD69KZT,La Butte Rouge,[Francis Marty],[2nuMRGzeJ5jJEKlfS7rZ0W]
7,0IA0Hju8CAgYfV1hwhidBH,La Java,[Mistinguett],[4AxgXfD7ISvJSTObqm4aIE]
8,0IgI1UCz84pYeVetnl1lGP,Old Fashioned Girl,[Greg Fieler],[5nWlsH5RDgFuRAiDeOFVmf]
9,0JV4iqw2lSKJaHBQZ0e5zK,Martín Fierro - Remasterizado,[Ignacio Corsini],[5LiOoJbxVSAMkBS2fUm3X2]


,spotify_track_id,spotify_artist_id
0,35iwgR4jXetI318WEWsa1Q,45tIt06XoI0Iio4LBEVpls
1,021ht4sdgPcrDgSk7JTbKY,14jtPCOoNZwquk5wd9DxrY
2,07A5yehtSnoedViJAZkNnc,5LiOoJbxVSAMkBS2fUm3X2
3,08FmqUhxtyLTn6pAh6bk45,5LiOoJbxVSAMkBS2fUm3X2
4,08y9GfoqCWfOGsKdwojr5e,3BiJGZsyX9sJchTqcSA7Su
5,0BRXJHRNGQ3W4v9frnSfhu,3BiJGZsyX9sJchTqcSA7Su
6,0Dd9ImXtAtGwsmsAD69KZT,2nuMRGzeJ5jJEKlfS7rZ0W
7,0IA0Hju8CAgYfV1hwhidBH,4AxgXfD7ISvJSTObqm4aIE
8,0IgI1UCz84pYeVetnl1lGP,5nWlsH5RDgFuRAiDeOFVmf
9,0JV4iqw2lSKJaHBQZ0e5zK,5LiOoJbxVSAMkBS2fUm3X2


Invalid IDs and duplicate track IDs are removed. Numeric audio validity is audited column by column; missing audio never removes a searchable song. Artist ID is the join key, so artists with the same name are not merged.

In [4]:
display(pd.read_csv(REPORTS / "catalog_audio_correlations.csv", index_col=0).round(3))

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,key,mode
danceability,1.000,0.242,0.251,0.199,-0.243,-0.226,-0.106,0.528,-0.049,-0.120,0.019,-0.045
energy,0.242,1.000,0.765,-0.054,-0.715,-0.196,0.125,0.372,0.228,0.025,0.036,-0.065
loudness,0.251,0.765,1.000,-0.167,-0.519,-0.329,0.030,0.275,0.185,0.000,0.027,-0.040
speechiness,0.199,-0.054,-0.167,1.000,0.069,-0.102,0.207,0.047,-0.089,-0.126,-0.001,-0.018
acousticness,-0.243,-0.715,-0.519,0.069,1.000,0.204,-0.005,-0.181,-0.196,-0.064,-0.027,0.059
instrumentalness,-0.226,-0.196,-0.329,-0.102,0.204,1.000,-0.039,-0.175,-0.053,0.069,-0.007,-0.010
liveness,-0.106,0.125,0.030,0.207,-0.005,-0.039,1.000,-0.000,-0.014,0.002,-0.007,0.007
valence,0.528,0.372,0.275,0.047,-0.181,-0.175,-0.000,1.000,0.131,-0.163,0.020,0.011
tempo,-0.049,0.228,0.185,-0.089,-0.196,-0.053,-0.014,0.131,1.000,-0.001,0.004,0.008
duration_ms,-0.120,0.025,0.000,-0.126,-0.064,0.069,0.002,-0.163,-0.001,1.000,0.005,-0.028
